# OSM exploratory analysis

### *Q?* what road network should we choose? Is there an official opensourced road network?
### *Quick Answer:* No, there's no official opensourced road network in Thailand from DOH, DRR.
- Department of Highway
- Department of Rural Highway

https://data.humdata.org/dataset/hotosm_tha_roads

https://download.geofabrik.de/asia/thailand-250902-free.shp.zip

Highway -> "trunk", "trunk_link",
    "primary", "primary_link",
    "secondary", "secondary_link"

Rural Highway -> "trunk", "trunk_link",
    "primary", "primary_link",
    "secondary", "secondary_link",
    "tertiary", "tertiary_link"


Important columns -> highway, osm_id

In [ ]:
import geopandas as gpd

road_network = gpd.read_file('../data/roadnetwork/hotosm_tha_roads_lines_shp/hotosm_tha_roads_lines_shp.shp')

In [ ]:
road_network.columns

In [ ]:
road_network['highway'].unique()

In [ ]:
road_network_TRAMS = road_network[road_network['highway'].isin([
    "trunk", "trunk_link",
    "primary", "primary_link",
    "secondary", "secondary_link",
    "tertiary", "tertiary_link"
])]

In [ ]:
road_network_TRAMS

In [ ]:
# how many % each
road_network_TRAMS_count = road_network_TRAMS['highway'].value_counts()
highway_percentages = (road_network_TRAMS_count / road_network_TRAMS_count.sum()) * 100

# print
# highway counts
print("Highway Counts:")
print(road_network_TRAMS_count)

# highway percentages
print("\nHighway Percentages:")
print(highway_percentages)

In [ ]:
road_network_TRAMS

In [ ]:
road_network_TRAMS = road_network_TRAMS.to_crs(epsg=32647)
# Calculate the length of each road segment in meters
road_network_TRAMS['length'] = road_network_TRAMS.geometry.length

# Group by highway class and sum the lengths
highway_distances = road_network_TRAMS.groupby('highway')['length'].sum()

# Convert to kilometers for better readability
highway_distances_km = highway_distances / 1000

# Print the distances
print("Highway Distances (in km):")
print(highway_distances_km)


# % by km
highway_distances_km_percent = (highway_distances_km / highway_distances_km.sum()) * 100
print("\nHighway Distances Percentages (by km):")
print(highway_distances_km_percent)


In [ ]:
# average, min, max distance of each road segment by type
by_type = road_network_TRAMS.groupby('highway')['length'].agg(['mean', 'min', 'max'])

print("Average length of each road segment (in km):", by_type['mean'] / 1000)
print("Minimum length of each road segment (in km):", by_type['min'] / 1000)
print("Maximum length of each road segment (in km):", by_type['max'] / 1000)


In [ ]:
# plot TRAMS on the map using basemap
import contextily as ctx
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 8))
road_network_TRAMS.to_crs(epsg=3857).plot(
    ax=ax, column='highway', legend=True, linewidth=1, alpha=0.7, cmap='tab10'
)
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, attribution_size=5, zorder=0)
ax.set_axis_off()
ax.legend(title='Highway', loc='lower right')


In [ ]:
import contextily as ctx
import matplotlib.pyplot as plt
import geopandas as gpd
import matplotlib.patches as mpatches
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
from matplotlib.font_manager import FontProperties

# ---------- helpers ----------
def _nice_length_m(data_width_m):
    """Pick a nice scalebar length (1/2/5 x 10^n) ~ 1/5 of axis width."""
    target = data_width_m / 5.0
    import math
    exp = int(math.floor(math.log10(target))) if target > 0 else 0
    base = target / (10 ** exp)
    for b in (1, 2, 5, 10):
        if base <= b:
            return b * (10 ** exp)
    return 10 * (10 ** exp)

def add_scalebar(ax, loc="lower left", font_size=9, pad=0.2):
    """Add a scalebar (meters) to an EPSG:3857 axis."""
    xmin, xmax = ax.get_xlim()
    length_m = _nice_length_m(xmax - xmin)
    label = f"{int(length_m/1000)} km" if length_m >= 1000 else f"{int(length_m)} m"
    fp = FontProperties(size=font_size)
    sb = AnchoredSizeBar(ax.transData, length_m, label, loc,
                         pad=pad, color='black', frameon=True,
                         size_vertical=(xmax - xmin) * 0.003,
                         fontproperties=fp)
    ax.add_artist(sb)

def add_north_arrow(ax, xy=(0.8, 0.82), size=0.08, text="N", text_size=10):
    """Add a simple north arrow in axes fraction coords."""
    ax.annotate("", xy=(xy[0], xy[1] + size), xytext=xy,
                xycoords="axes fraction", textcoords="axes fraction",
                arrowprops=dict(arrowstyle="-|>", linewidth=1.5, color="black"))
    ax.text(xy[0], xy[1] + size + 0.015, text,
            transform=ax.transAxes, ha="center", va="bottom",
            fontsize=text_size, fontweight="bold")

# ---------- categorical plotting ----------
unique_vals = road_network_TRAMS['highway'].unique()
colors = plt.cm.tab10.colors  # 10-color palette
color_map = {val: colors[i % len(colors)] for i, val in enumerate(unique_vals)}

fig, ax = plt.subplots(figsize=(12, 8))

for val, color in color_map.items():
    subset = road_network_TRAMS[road_network_TRAMS['highway'] == val].to_crs(epsg=3857)
    subset.plot(ax=ax, color=color, linewidth=1, alpha=0.7, label=val)

# basemap
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, attribution_size=5, zorder=0)

# legend
handles = [mpatches.Patch(color=color, label=val) for val, color in color_map.items()]
ax.legend(handles=handles, title="Highway type", loc="lower right", bbox_to_anchor=(1, 0.1))

# scale + north arrow
add_scalebar(ax, loc="lower right")
add_north_arrow(ax, xy=(0.95, 0.85), size=0.08)

ax.set_axis_off()
plt.tight_layout()
plt.savefig('fig/TRAMS_road_network.png', dpi=300, bbox_inches='tight', pad_inches=0)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import contextily as ctx
import geopandas as gpd
import matplotlib.patches as mpatches
from pathlib import Path

# --- If you don't already have these helpers, uncomment this block ---
# from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
# from matplotlib.font_manager import FontProperties
# import math
# def _nice_length_m(data_width_m):
#     target = data_width_m / 5.0
#     exp = int(math.floor(math.log10(target))) if target > 0 else 0
#     base = target / (10 ** exp)
#     for b in (1, 2, 5, 10):
#         if base <= b:
#             return b * (10 ** exp)
#     return 10 * (10 ** exp)
# def add_scalebar(ax, loc="lower left", font_size=9, pad=0.2):
#     xmin, xmax = ax.get_xlim()
#     length_m = _nice_length_m(max(xmax - xmin, 1))
#     label = f"{int(length_m/1000)} km" if length_m >= 1000 else f"{int(length_m)} m"
#     sb = AnchoredSizeBar(ax.transData, length_m, label, loc,
#                          pad=pad, color='black', frameon=True,
#                          size_vertical=max((xmax - xmin), 1) * 0.003,
#                          fontproperties=FontProperties(size=font_size))
#     ax.add_artist(sb)
# def add_north_arrow(ax, xy=(0.08, 0.82), size=0.08, text="N", text_size=10):
#     ax.annotate("", xy=(xy[0], xy[1] + size), xytext=xy,
#                 xycoords="axes fraction", textcoords="axes fraction",
#                 arrowprops=dict(arrowstyle="-|>", linewidth=1.5, color="black"))
#     ax.text(xy[0], xy[1] + size + 0.015, text,
#             transform=ax.transAxes, ha="center", va="bottom",
#             fontsize=text_size, fontweight="bold")

# ---------- config ----------
pairs = [
    ("motorway",  "motorway_link"),
    ("trunk",     "trunk_link"),
    ("primary",   "primary_link"),
    ("secondary", "secondary_link"),
    ("tertiary",  "tertiary_link"),
]
main_color = "tab:blue"
link_color = "tab:orange"
outdir = Path("fig/")
outdir.mkdir(parents=True, exist_ok=True)

# ---------- prep ----------
rn_3857 = road_network_TRAMS.to_crs(3857)
all_bounds = rn_3857.total_bounds  # for consistent extent
xmin, ymin, xmax, ymax = all_bounds

for main, link in pairs:
    sub_main = rn_3857[rn_3857["highway"] == main]
    sub_link = rn_3857[rn_3857["highway"] == link]

    if sub_main.empty and sub_link.empty:
        # nothing to plot for this pair
        continue

    fig, ax = plt.subplots(figsize=(10, 7))

    # plot main first (thicker), then link (thinner)
    if not sub_main.empty:
        sub_main.plot(ax=ax, color=main_color, linewidth=1.4, alpha=0.9, zorder=3)
    if not sub_link.empty:
        sub_link.plot(ax=ax, color=link_color, linewidth=1.0, alpha=0.9, zorder=3)

    # basemap + consistent extent
    ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, attribution_size=5, zorder=0)
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_axis_off()

    # legend (inside map, lower-right; nudge up with bbox_to_anchor)
    handles = []
    if not sub_main.empty:
        handles.append(mpatches.Patch(color=main_color, label=main.title()))
    if not sub_link.empty:
        handles.append(mpatches.Patch(color=link_color, label=link.replace("_", " ").title()))
    ax.legend(handles=handles, title="Highway class",
              loc="lower right", bbox_to_anchor=(1.0, 0.20), frameon=True)

    # north arrow + scalebar
    add_north_arrow(ax, xy=(0.95, 0.85), size=0.08)
    add_scalebar(ax, loc="lower right")

    # title = f"{main.title()} & {link.replace('_', ' ').title()}"
    # ax.set_title(title, fontsize=13, pad=6)

    # save
    fname = f"pair_{main}.png"
    fig.savefig(outdir / fname, dpi=300, bbox_inches="tight", pad_inches=0)
    plt.show()
    plt.close(fig)


In [ ]:
#plot missingness of road network trams


missingness = road_network_TRAMS.isna().mean()
missingness.plot(kind='bar')
plt.title('Missingness of Road Network Trams')
plt.xlabel('Attributes')
plt.ylabel('Missingness Proportion')
plt.show()


In [ ]:
for column in road_network_TRAMS.columns:
    unique_values = road_network_TRAMS[column].unique()
    print(f"Unique values in column '{column}':")
    print(unique_values)
    print()

# Thailand: Road Surface Data
https://data.humdata.org/dataset/thailand-road-surface-data

In [ ]:
roadsurface_data = gpd.read_file('roadnetwork/heigit_tha_roadsurface_lines.gpkg')
roadsurface_data.head()

In [ ]:
roadsurface_data_TRAMS = roadsurface_data[roadsurface_data['highway'].isin([
    "trunk", "trunk_link",
    "primary", "primary_link",
    "secondary", "secondary_link",
    "tertiary", "tertiary_link"
])]

In [ ]:
# how many % each
road_network_TRAMS_count = road_network_TRAMS['highway'].value_counts()
highway_percentages = (road_network_TRAMS_count / road_network_TRAMS_count.sum()) * 100

# print
# highway counts
print("Highway Counts:")
print(road_network_TRAMS_count)

# highway percentages
print("\nHighway Percentages:")
print(highway_percentages)


# how many % each
roadsurface_data_TRAMS_count = roadsurface_data_TRAMS['highway'].value_counts()
roadsurface_highway_percentages = (roadsurface_data_TRAMS_count / roadsurface_data_TRAMS_count.sum()) * 100

# print
# highway counts
print("Highway Counts:")
print(roadsurface_data_TRAMS_count)

# highway percentages
print("\nHighway Percentages:")
print(roadsurface_highway_percentages)


In [ ]:
missingness = roadsurface_data_TRAMS.isna().mean()
missingness.plot(kind='bar')
plt.title('Missingness of Road Network Trams')
plt.xlabel('Attributes')
plt.ylabel('Missingness Proportion')
plt.show()


In [ ]:
print(roadsurface_data_TRAMS.columns)
print(len(roadsurface_data_TRAMS['osm_id'].unique()))

# OSM
../data/roadnetwork/thailand-latest-free.shp/gis_osm_roads_free_1.shp

In [ ]:
osm = gpd.read_file("roadnetwork/thailand-250902-free.shp/gis_osm_roads_free_1.shp")

In [ ]:
osm.columns

In [ ]:
osm_TRAMS = osm[osm['fclass'].isin([
    "trunk", "trunk_link",
    "primary", "primary_link",
    "secondary", "secondary_link",
    "tertiary", "tertiary_link"
])]
osm_TRAMS

In [ ]:
osm_TRAMS = osm_TRAMS[osm_TRAMS['ref'].notnull()]
osm_TRAMS

In [ ]:
missingness = osm_TRAMS.isna().mean()
missingness.plot(kind='bar')
plt.title('Missingness of Road Network Trams')
plt.xlabel('Attributes')
plt.ylabel('Missingness Proportion')
plt.show()

In [ ]:
# unique osm_id
print('osm_TRAMS:', len(osm_TRAMS['osm_id'].unique()))
print('road_network_TRAMS:', len(road_network_TRAMS['osm_id'].unique()))
print('roadsurface_data_TRAMS:', len(roadsurface_data_TRAMS['osm_id'].unique()))

In [ ]:
# Ensure dtype consistency for osm_id columns
osm_TRAMS['osm_id'] = osm_TRAMS['osm_id'].astype(str)
road_network_TRAMS['osm_id'] = road_network_TRAMS['osm_id'].astype(str)
roadsurface_data_TRAMS['osm_id'] = roadsurface_data_TRAMS['osm_id'].astype(str)

# Unique osm_id sets
osm_ids_osm_TRAMS = set(osm_TRAMS['osm_id'].unique())
osm_ids_road_network_TRAMS = set(road_network_TRAMS['osm_id'].unique())
osm_ids_roadsurface_data_TRAMS = set(roadsurface_data_TRAMS['osm_id'].unique())

# Compare osm_id between the three sources
common_ids_all = osm_ids_osm_TRAMS & osm_ids_road_network_TRAMS & osm_ids_roadsurface_data_TRAMS
common_ids_osm_road_network = osm_ids_osm_TRAMS & osm_ids_road_network_TRAMS
common_ids_osm_roadsurface = osm_ids_osm_TRAMS & osm_ids_roadsurface_data_TRAMS
common_ids_road_network_roadsurface = osm_ids_road_network_TRAMS & osm_ids_roadsurface_data_TRAMS

# Print results
print(f"Total unique osm_id in osm_TRAMS: {len(osm_ids_osm_TRAMS)}")
print(f"Total unique osm_id in road_network_TRAMS: {len(osm_ids_road_network_TRAMS)}")
print(f"Total unique osm_id in roadsurface_data_TRAMS: {len(osm_ids_roadsurface_data_TRAMS)}\n")

print(f"Common osm_id in all three sources: {len(common_ids_all)}")
print(f"Common osm_id between osm_TRAMS and road_network_TRAMS: {len(common_ids_osm_road_network)}")
print(f"Common osm_id between osm_TRAMS and roadsurface_data_TRAMS: {len(common_ids_osm_roadsurface)}")
print(f"Common osm_id between road_network_TRAMS and roadsurface_data_TRAMS: {len(common_ids_road_network_roadsurface)}")

In [ ]:
common_ids_road_network_roadsurface

In [ ]:
road_network_TRAMS['osm_id'].dtype

In [ ]:
# we will be using road_network_TRAMS just match osm_id with osm_TRAMS to get the col ref

road_network_TRAMS = road_network_TRAMS.merge(
    osm_TRAMS[['osm_id', 'ref']], on='osm_id', how='left'
)

road_network_TRAMS

In [ ]:
# how many road_network_TRAMS['ref'] isnotnull

road_network_TRAMS['ref_x'].notnull().sum()

In [ ]:
#save road_network_TRAMS
road_network_TRAMS.to_file("roadnetwork/road_network_TRAMS.geojson", driver='GeoJSON')

# Road Network
### 1. Highway class filter
Selecting only trunk, primary, secondary, tertiary (and their _link variants) is aligned with OSM tagging conventions and roughly maps to Thailand’s DOH/DRR highway networks.

### 2. Using ref
Requiring a non-null ref ensures you’re working with officially numbered roads (national or rural highways). This is critical - otherwise you’d be including urban arterials, service roads, or tertiary streets that may not be part of the “formal” highway system.

### 3. Spatial clipping with incidents
Buffering historical incident points (10 m) and clipping the road network ensures:


https://data.humdata.org/dataset/hotosm_tha_roads

https://download.geofabrik.de/asia/thailand-250902-free.shp.zip

In [ ]:
import geopandas as gpd
import pandas as pd

road_network = gpd.read_file('roadnetwork/hotosm_tha_roads_lines_shp/hotosm_tha_roads_lines_shp.shp') # humdata
osm = gpd.read_file("roadnetwork/thailand-250902-free.shp/gis_osm_roads_free_1.shp") # geofabrik
accidents = gpd.read_file("DOH/motorcycle_accidents_TRAMS.shp") # accidents from TRAMS

In [ ]:
# --- pick a metric CRS for Thailand (UTM 47N suits BKK/central; use 48N if far east) ---
TARGET_CRS = "EPSG:32647"
road_network = road_network.to_crs(TARGET_CRS)
osm          = osm.to_crs(TARGET_CRS)
accidents    = accidents.to_crs(TARGET_CRS)

In [ ]:
# --- filter highway classes ---
allowed = {
    "trunk","trunk_link",
    "primary","primary_link",
    "secondary","secondary_link",
    "tertiary","tertiary_link"
}
roads_osm  = osm[osm["fclass"].isin(allowed)].copy()
roads_hot  = road_network[road_network["highway"].isin(allowed)].copy()

In [ ]:
# --- attach 'ref' from Geofabrik to HOTOSM by osm_id (if available) ---
roads_hot["osm_id"] = roads_hot["osm_id"].astype(str)
roads_osm["osm_id"] = roads_osm["osm_id"].astype(str)

roads_hot = roads_hot.merge(
    roads_osm[["osm_id","ref"]],
    on="osm_id", how="left", validate="m:1"
)

In [ ]:
# --- keep only numbered routes (non-null/non-empty ref) ---
roads_hot["ref"] = roads_hot["ref"].astype(str).str.strip()
roads_num = roads_hot.loc[roads_hot["ref"].ne("") & (roads_hot["ref"].str.lower() != "nan")].copy()


In [ ]:
# --- build 100 m accident buffer (metric CRS) ---
acc_buf = accidents.copy()
acc_buf["geometry"] = acc_buf.geometry.buffer(100)

# --- intersect: CLIP (cuts lines to inside-buffer pieces) ---
mask = gpd.GeoDataFrame(geometry=[acc_buf.unary_union], crs=acc_buf.crs)
study_network = gpd.clip(roads_num, mask)

In [ ]:
# --- optional sanity checks ---
print("roads_num:", len(roads_num), "study_network:", len(study_network))
print("sample refs:", study_network["ref"].dropna().astype(str).head().tolist())

In [ ]:
road_network_TRAMS = road_network[road_network['highway'].isin([
    "trunk", "trunk_link",
    "primary", "primary_link",
    "secondary", "secondary_link",
    "tertiary", "tertiary_link"
])]

osm_TRAMS = osm[osm['fclass'].isin([
    "trunk", "trunk_link",
    "primary", "primary_link",
    "secondary", "secondary_link",
    "tertiary", "tertiary_link"
])]


In [ ]:
# Display the columns and counts for road_network_TRAMS and osm_TRAMS
print("Road Network TRAMS Columns:")
print(road_network_TRAMS.columns)
print("\nRoad Network TRAMS Count:")
print(road_network_TRAMS.count())

print("\nOSM TRAMS Columns:")
print(osm_TRAMS.columns)
print("\nOSM TRAMS Count:")
print(osm_TRAMS.count())

In [ ]:
# Ensure dtype consistency for 'osm_id' columns
road_network_TRAMS['osm_id'] = road_network_TRAMS['osm_id'].astype(str)
osm_TRAMS['osm_id'] = osm_TRAMS['osm_id'].astype(str)

# Merge road_network_TRAMS with osm_TRAMS to get the 'ref' column
road_network_TRAMS = road_network_TRAMS.merge(
    osm_TRAMS[['osm_id', 'ref']], on='osm_id', how='left'
)

road_network_TRAMS

In [ ]:
mask = gpd.GeoDataFrame(geometry=[accidents_buffer.unary_union], crs=target_crs)
roads_clip = gpd.clip(road_network_TRAMS[road_network_TRAMS['ref'].isna()], mask)

In [ ]:
road_network_TRAMS

In [ ]:
mask = gpd.GeoDataFrame(geometry=[accidents_buffer.unary_union], crs=target_crs)
roads_clip = gpd.clip(roads_num, mask)

In [ ]:
import geopandas as gpd
import pandas as pd

# --------------------------
# Config
# --------------------------
# Use UTM 47N for Bangkok/central/west Thailand; switch to "EPSG:32648" for far east.
TARGET_CRS = "EPSG:32647"
BUF_M = 100

ALLOWED = {
    "trunk","trunk_link",
    "primary","primary_link",
    "secondary","secondary_link",
    "tertiary","tertiary_link"
}

# --------------------------
# 0) Read data
# --------------------------
road_network = gpd.read_file("roadnetwork/hotosm_tha_roads_lines_shp/hotosm_tha_roads_lines_shp.shp")  # HOTOSM
osm          = gpd.read_file("roadnetwork/thailand-250902-free.shp/gis_osm_roads_free_1.shp")          # Geofabrik
accidents    = gpd.read_file("DOH/motorcycle_accidents_TRAMS.shp")                                     # TRAMS

# --------------------------
# 1) Reproject to metric CRS
# --------------------------
road_network = road_network.to_crs(TARGET_CRS)
osm          = osm.to_crs(TARGET_CRS)
accidents    = accidents.to_crs(TARGET_CRS)

# --------------------------
# 2) Filter highway classes
# --------------------------
# HOTOSM typically uses 'highway'
if "highway" not in road_network.columns:
    raise KeyError("Expected 'highway' column in HOTOSM roads.")

roads_hot = road_network[road_network["highway"].isin(ALLOWED)].copy()

# Geofabrik uses 'fclass' for functional class; fall back to 'highway' if needed.
osm_class_col = "fclass" if "fclass" in osm.columns else ("highway" if "highway" in osm.columns else None)
if osm_class_col is None:
    raise KeyError("Expected 'fclass' or 'highway' column in Geofabrik roads.")

roads_osm = osm[osm[osm_class_col].isin(ALLOWED)].copy()

# --------------------------
# 3) Attach 'ref' to HOTOSM roads
#    Prefer fast key merge via 'osm_id'; if not possible, use nearest spatial join
# --------------------------
if "osm_id" in roads_hot.columns and "osm_id" in roads_osm.columns:
    # Ensure dtype consistency
    roads_hot["osm_id"] = roads_hot["osm_id"].astype(str)
    roads_osm["osm_id"] = roads_osm["osm_id"].astype(str)
    # Merge (many HOTOSM features may not match Geofabrik 1:1; that's okay)
    roads_hot = roads_hot.merge(
        roads_osm[["osm_id", "ref"]].copy(),
        on="osm_id", how="left", validate="m:1"
    )
else:
    # Fallback: nearest join within small tolerance to bring 'ref' across
    # Keep only ref + geometry on the right to avoid column collisions.
    right = roads_osm[["ref", "geometry"]].copy()
    roads_hot = gpd.sjoin_nearest(
        roads_hot, right,
        how="left",
        max_distance=20,   # meters; adjust if necessary
        distance_col="join_dist"
    ).drop(columns=[c for c in ["index_right"] if c in roads_hot.columns])

# --------------------------
# 4) Filter to numbered routes (non-null, non-empty ref)
# --------------------------
roads_hot["ref"] = roads_hot["ref"].astype(str).str.strip()
roads_num = roads_hot.loc[
    roads_hot["ref"].ne("") & (roads_hot["ref"].str.lower() != "nan")
].copy()

# --------------------------
# 5) Build 100 m accident buffers
# --------------------------
acc_buf = accidents.copy()
acc_buf["geometry"] = acc_buf.geometry.buffer(BUF_M)

# Single unioned mask polygon (Shapely 2 preferred API)
try:
    mask_geom = acc_buf.union_all()         # GeoPandas >=0.14 / Shapely 2
except AttributeError:
    mask_geom = acc_buf.unary_union         # fallback for older versions

# --------------------------
# 6A) CLIP to buffers (cuts lines)
# --------------------------
study_clip = gpd.clip(roads_num, mask_geom)

# --------------------------
# 6B) SELECT whole segments intersecting buffers (keeps full geometry)
# --------------------------
sel_idx = gpd.sjoin(
    roads_num[["geometry"]],
    acc_buf[["geometry"]],
    predicate="intersects", how="inner"
).index.unique()
study_select = roads_num.loc[sel_idx].copy()

# --------------------------
# 7) Quick sanity checks
# --------------------------
print(f"Total HOTOSM highways (allowed classes): {len(roads_hot)}")
print(f"Numbered routes after ref filter:        {len(roads_num)}")
print(f"Clipped segments within {BUF_M} m:        {len(study_clip)}")
print(f"Whole segments intersecting {BUF_M} m:    {len(study_select)}")
print("Sample refs:", study_select["ref"].head(5).tolist())

# --------------------------
# 8) (Optional) Export
# --------------------------
# study_clip.to_file("out/study_network_clip_100m.shp")
# study_select.to_file("out/study_network_select_100m.shp")


In [ ]:
import geopandas as gpd
import pandas as pd

# --------------------------
# Config
# --------------------------
TARGET_CRS = "EPSG:32647"   # use 32648 for far east Thailand if needed
BUF_ADD_M  = 500            # for adding ref-null roads near accidents
ALLOWED = {
    "trunk","trunk_link",
    "primary","primary_link",
    "secondary","secondary_link",
    "tertiary","tertiary_link"
}

# --------------------------
# 0) Read data
# --------------------------
road_network = gpd.read_file("roadnetwork/hotosm_tha_roads_lines_shp/hotosm_tha_roads_lines_shp.shp")  # HOTOSM
osm          = gpd.read_file("roadnetwork/thailand-250902-free.shp/gis_osm_roads_free_1.shp")          # Geofabrik
accidents    = gpd.read_file("DOH/motorcycle_accidents_TRAMS.shp")                                     # TRAMS

# --------------------------
# 1) Reproject to metric CRS
# --------------------------
road_network = road_network.to_crs(TARGET_CRS)
osm          = osm.to_crs(TARGET_CRS)
accidents    = accidents.to_crs(TARGET_CRS)

# --------------------------
# 2) Filter highway classes
# --------------------------
if "highway" not in road_network.columns:
    raise KeyError("Expected 'highway' column in HOTOSM roads.")
roads_hot = road_network[road_network["highway"].isin(ALLOWED)].copy()

osm_class_col = "fclass" if "fclass" in osm.columns else ("highway" if "highway" in osm.columns else None)
if osm_class_col is None:
    raise KeyError("Expected 'fclass' or 'highway' column in Geofabrik roads.")
roads_osm = osm[osm[osm_class_col].isin(ALLOWED)].copy()

# --------------------------
# 3) Attach 'ref' to HOTOSM roads (prefer merge on osm_id; fallback to nearest)
# --------------------------
if "osm_id" in roads_hot.columns and "osm_id" in roads_osm.columns:
    roads_hot["osm_id"] = roads_hot["osm_id"].astype(str)
    roads_osm["osm_id"] = roads_osm["osm_id"].astype(str)
    roads_hot = roads_hot.merge(
        roads_osm[["osm_id","ref"]].copy(),
        on="osm_id", how="left", validate="m:1"
    )
else:
    right = roads_osm[["ref","geometry"]].copy()
    roads_hot = gpd.sjoin_nearest(
        roads_hot, right,
        how="left",
        max_distance=20,     # meters; raise if needed
        distance_col="join_dist"
    ).drop(columns=[c for c in ["index_right"] if c in roads_hot.columns])

# --------------------------
# 4) Split by ref presence
# --------------------------
ref_valid_mask = roads_hot["ref"].notna() & (roads_hot["ref"].astype(str).str.strip() != "")
roads_num      = roads_hot.loc[ref_valid_mask].copy()      # keep ALL of these
roads_ref_null = roads_hot.loc[~ref_valid_mask].copy()     # consider adding near accidents only

# --------------------------
# 5) Build 500 m buffers around accidents
# --------------------------
acc_buf500 = accidents.copy()
acc_buf500["geometry"] = acc_buf500.geometry.buffer(BUF_ADD_M)

# --------------------------
# 6) Add only ref-null roads within 500 m of accidents (keep whole segments)
# --------------------------
near_idx = gpd.sjoin(
    roads_ref_null[["geometry"]],
    acc_buf500[["geometry"]],
    predicate="intersects", how="inner"
).index.unique()
roads_ref_null_near = roads_ref_null.loc[near_idx].copy()

# --------------------------
# 7) Combine: (ALL ref-present) ∪ (ref-null within 500 m)
# --------------------------
study_select = pd.concat([roads_num, roads_ref_null_near], ignore_index=True)

# De-duplicate
if "osm_id" in study_select.columns:
    study_select["osm_id"] = study_select["osm_id"].astype(str)
    study_select = study_select.drop_duplicates(subset="osm_id")
else:
    # fallback dedupe by geometry
    study_select["__wkb__"] = study_select.geometry.apply(lambda g: g.wkb)
    study_select = study_select.drop_duplicates(subset="__wkb__").drop(columns="__wkb__")

# --------------------------
# 8) Sanity checks
# --------------------------
print(f"Kept ALL numbered routes (ref present): {len(roads_num)}")
print(f"Added ref-null roads within {BUF_ADD_M} m: {len(roads_ref_null_near)}")
print(f"Final study network (unique): {len(study_select)}")
print("Sample refs:", study_select["ref"].dropna().astype(str).head(5).tolist())

# --------------------------
# 9) (Optional) Export
# --------------------------
# study_select.to_file("out/study_network_all_ref_plus_null_500m.shp")


# Use below for road network

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# provinces = Path("../data/boundary/TH_Province.shp")
provinces = gpd.read_file("../data/boundary/TH_Province.shp")

# check the CRS
provinces.crs

In [ ]:
import geopandas as gpd
import pandas as pd
import re

# --------------------------
# Config
# --------------------------
TARGET_CRS      = "EPSG:32647"   # use 32648 if the AOI is far-east Thailand
BUF_ADD_M       = 500            # add ref-null roads within this distance of accidents
BUF_ANALYSIS_M  = 100            # clip analysis geometry to this distance

ALLOWED = {
    "trunk","trunk_link",
    "primary","primary_link",
    "secondary","secondary_link",
    "tertiary","tertiary_link"
}

# Regex: numeric refs (DOH) and Thai-letter refs (DRR)
RE_NUMERIC = re.compile(r"^\s*\d{1,4}\s*$")
RE_THAI_DRR = re.compile(r"^\s*[ก-๙]{1,3}\.?\d{2,5}\s*$")

def normalize_ref(r):
    """Return a single clean ref token (numeric or Thai-DRR) from multi-valued refs."""
    if r is None: 
        return None
    tokens = re.split(r"[;,/|]\s*", str(r))
    for t in tokens:
        t = t.strip()
        if RE_THAI_DRR.fullmatch(t) or RE_NUMERIC.fullmatch(t):
            return t
    return tokens[0].strip() if tokens and tokens[0].strip() else None

def owner_from_ref(r):
    if r is None: return "unknown"
    if RE_NUMERIC.fullmatch(r): return "DOH"   # national highways
    if RE_THAI_DRR.fullmatch(r): return "DRR"  # rural/provincial roads
    return "unknown"

# --------------------------
# 0) Read (Geofabrik roads + TRAMS accidents)
# --------------------------
osm = gpd.read_file("roadnetwork/thailand-250902-free.shp/gis_osm_roads_free_1.shp")
acc = gpd.read_file("DOH/motorcycle_accidents_TRAMS.shp")

# --------------------------
# 1) Reproject to metric CRS
# --------------------------
osm = osm.to_crs(TARGET_CRS)
acc = acc.to_crs(TARGET_CRS)

# --------------------------
# 2) Filter highway classes
# --------------------------
cls_col = "fclass" if "fclass" in osm.columns else ("highway" if "highway" in osm.columns else None)
if cls_col is None:
    raise KeyError("Expected 'fclass' or 'highway' in Geofabrik roads.")
roads = osm[osm[cls_col].isin(ALLOWED)].copy()

# --------------------------
# 3) Clean ref + owner label
# --------------------------
roads["ref"] = roads["ref"].apply(normalize_ref)
roads["is_numbered"] = roads["ref"].notna()
roads["owner"] = roads["ref"].apply(owner_from_ref)

# --------------------------
# 4) Split by ref presence
# --------------------------
roads_num      = roads.loc[roads["is_numbered"]].copy()   # keep ALL of these
roads_ref_null = roads.loc[~roads["is_numbered"]].copy()  # add near accidents only

# --------------------------
# 5) Build buffers
# --------------------------
acc_buf500 = acc.copy();  acc_buf500["geometry"] = acc_buf500.geometry.buffer(BUF_ADD_M)
acc_buf100 = acc.copy();  acc_buf100["geometry"] = acc_buf100.geometry.buffer(BUF_ANALYSIS_M)

# --------------------------
# 6) Add ref-null roads within 500 m (keep whole segments)
# --------------------------
near_idx = gpd.sjoin(
    roads_ref_null[["geometry"]],
    acc_buf500[["geometry"]],
    predicate="intersects",
    how="inner"
).index.unique()
roads_ref_null_near = roads_ref_null.loc[near_idx].copy()
roads_ref_null_near["selected_by_distance"] = True

# --------------------------
# 7) Combine coverage = (all numbered) ∪ (ref-null within 500 m)
# --------------------------
coverage = pd.concat([roads_num, roads_ref_null_near], ignore_index=True)

# Deduplicate
if "osm_id" in coverage.columns:
    coverage["osm_id"] = coverage["osm_id"].astype(str)
    coverage = coverage.drop_duplicates(subset="osm_id")
else:
    coverage["__wkb__"] = coverage.geometry.apply(lambda g: g.wkb)
    coverage = coverage.drop_duplicates(subset="__wkb__").drop(columns="__wkb__")

coverage["length_m"] = coverage.geometry.length
coverage["selected_by_distance"] = coverage.get("selected_by_distance", False)

# --------------------------
# 8) Analysis layer: clip coverage to 100 m buffers
# --------------------------
try:
    mask100 = acc_buf100.union_all()   # GeoPandas >=0.14 / Shapely 2
except AttributeError:
    mask100 = acc_buf100.unary_union

analysis_clip = gpd.clip(coverage, mask100)
analysis_clip["length_m"] = analysis_clip.geometry.length
analysis_clip["in_100m"] = True

# --------------------------
# 9) Checks
# --------------------------
print(f"ALL numbered routes kept: {len(roads_num)}")
print(f"Added ref-null within {BUF_ADD_M} m: {len(roads_ref_null_near)}")
print(f"Coverage layer (unique): {len(coverage)}")
print(f"Analysis clip (<={BUF_ANALYSIS_M} m): {len(analysis_clip)}")
print("Owner breakdown (coverage):")
print(coverage["owner"].value_counts(dropna=False))

# --------------------------
# 10) Optional export
# --------------------------
# coverage.to_file("out/tha_highway_coverage.gpkg", layer="coverage", driver="GPKG")
# analysis_clip.to_file("out/tha_highway_analysis_100m.gpkg", layer="analysis_100m", driver="GPKG")


In [ ]:
coverage

In [ ]:
accidents    = gpd.read_file("../data/DOH/motorcycle_accidents_TRAMS.shp")                                     # TRAMS


In [ ]:
coverage = gpd.read_file("../data/roadnetwork/tha_highway_coverage.gpkg", layer="coverage", driver="GPKG")

In [ ]:
coverage

In [ ]:
# clip coverage using provinces
coverage = gpd.clip(coverage, provinces)

In [ ]:
coverage.to_file("roadnetwork/tha_highway_coverage_clip.gpkg", layer="coverage", driver="GPKG")
# analysis_clip.to_file("roadnetwork/tha_highway_analysis_100m.gpkg", layer="analysis_100m", driver="GPKG")


In [ ]:
#print all unique ref
add_ref = coverage['ref'].unique()
print(add_ref)

In [ ]:
# Clean up the references: strip spaces
refs = pd.Series(add_ref).astype(str).str.strip()

# Identify patterns
non_numeric = refs[~refs.str.replace(" ", "").str.isnumeric()]
with_spaces = refs[refs.str.contains(" ")]
short_codes = refs[refs.str.len() < 3]
long_codes = refs[refs.str.len() > 6]

# Summarize odd cases
odd_summary = {
    "total_refs": len(refs),
    "non_numeric": non_numeric.tolist()[:20],  # show sample
    "with_spaces": with_spaces.tolist()[:20],
    "short_codes": short_codes.tolist()[:20],
    "long_codes": long_codes.tolist()[:20]
}

odd_summary


In [ ]:
non_numeric

In [ ]:
# update length
coverage['length_m'] = coverage.geometry.length

In [ ]:
# Group by highway class and sum the lengths
highway_distances = coverage.groupby('fclass')['length_m'].sum()

# Convert to kilometers for better readability
highway_distances_km = highway_distances / 1000

# Print the distances
print("Highway Distances (in km):")
print(highway_distances_km)


# % by km
highway_distances_km_percent = (highway_distances_km / highway_distances_km.sum()) * 100
print("\nHighway Distances Percentages (by km):")
print(highway_distances_km_percent)


In [ ]:

# count each fclass in coverage... how many accident count in each fclass
accidents = gpd.read_file("../data/DOH/motorcycle_accidents_TRAMS.shp")
# Ensure both GeoDataFrames use the same CRS
accidents = accidents.to_crs(coverage.crs)



In [ ]:
accidents

In [ ]:
# Spatial join to count accidents per highway class >> map to closest highway segment (nearest)
accidents_with_fclass = gpd.sjoin_nearest(accidents, coverage[["fclass", "geometry"]], how="left", distance_col="distance")
accident_counts = accidents_with_fclass.groupby("fclass").size()
print("\nAccident Counts by Highway Class:")
print(accident_counts)


In [ ]:
accidents_with_fclass = gpd.sjoin_nearest(
    accidents,
    coverage[["fclass", "geometry"]],
    how="left",
    distance_col="distance"
)

accidents_with_fclass = (
    accidents_with_fclass
    .reset_index()
    .sort_values(["index", "distance"])
    .drop_duplicates(subset="index", keep="first")
    .set_index("index")
)

accident_counts = accidents_with_fclass.groupby("fclass").size()
print(accident_counts)
print(accident_counts.sum())  # should now be 29539, unless some unmatched rows exist

In [ ]:
# Spatial join to count accidents per highway class >> map to closest highway segment (nearest)
accidents_with_fclass = gpd.sjoin_nearest(accidents, coverage[["fclass", "geometry"]], how="left", distance_col="distance")

# Drop duplicates based on accident index to avoid overcounting
accidents_with_fclass = accidents_with_fclass[~accidents_with_fclass.index.duplicated(keep="first")]

accident_counts = accidents_with_fclass.groupby("fclass").size()
print("\nAccident Counts by Highway Class:")
print(accident_counts)
print(f"Total accidents after deduplication: {len(accidents_with_fclass)}")


In [ ]:
# average, min, max distance of each road segment by type
by_type = coverage.groupby('fclass')['length_m'].agg(['mean', 'min', 'max'])

print("Average length of each road segment (in km):", by_type['mean'] / 1000)
print("Minimum length of each road segment (in m):", by_type['min'])
print("Maximum length of each road segment (in km):", by_type['max'] / 1000)


In [ ]:
# sum of distances when ref is not null
sum_distances_notnull = coverage[coverage["ref"].notnull()]["length_m"].sum()/1000
print(sum_distances_notnull)
sum_distances = coverage["length_m"].sum()/1000
print(sum_distances)

In [ ]:
# coverage['fclass'] how many rows unique each?
print("Number of unique road classes:", coverage['fclass'].nunique())
print("Number of rows for each road class:")
print(coverage['fclass'].value_counts())

In [ ]:
# plot TRAMS on the map using basemap
import contextily as ctx
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 8))
coverage.to_crs(epsg=3857).plot(
    ax=ax, column='fclass', legend=True, linewidth=1, alpha=0.7, cmap='tab10'
)
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, attribution_size=5, zorder=0)
ax.set_axis_off()
ax.legend(title='Highway', loc='lower right')


In [ ]:
import contextily as ctx
import matplotlib.pyplot as plt
import geopandas as gpd
import matplotlib.patches as mpatches
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
from matplotlib.font_manager import FontProperties

# ---------- helpers ----------
def _nice_length_m(data_width_m):
    """Pick a nice scalebar length (1/2/5 x 10^n) ~ 1/5 of axis width."""
    target = data_width_m / 5.0
    import math
    exp = int(math.floor(math.log10(target))) if target > 0 else 0
    base = target / (10 ** exp)
    for b in (1, 2, 5, 10):
        if base <= b:
            return b * (10 ** exp)
    return 10 * (10 ** exp)

def add_scalebar(ax, loc="lower left", font_size=9, pad=0.2):
    """Add a scalebar (meters) to an EPSG:3857 axis."""
    xmin, xmax = ax.get_xlim()
    length_m = _nice_length_m(xmax - xmin)
    label = f"{int(length_m/1000)} km" if length_m >= 1000 else f"{int(length_m)} m"
    fp = FontProperties(size=font_size)
    sb = AnchoredSizeBar(ax.transData, length_m, label, loc,
                         pad=pad, color='black', frameon=True,
                         size_vertical=(xmax - xmin) * 0.003,
                         fontproperties=fp)
    ax.add_artist(sb)

def add_north_arrow(ax, xy=(0.8, 0.82), size=0.08, text="N", text_size=10):
    """Add a simple north arrow in axes fraction coords."""
    ax.annotate("", xy=(xy[0], xy[1] + size), xytext=xy,
                xycoords="axes fraction", textcoords="axes fraction",
                arrowprops=dict(arrowstyle="-|>", linewidth=1.5, color="black"))
    ax.text(xy[0], xy[1] + size + 0.015, text,
            transform=ax.transAxes, ha="center", va="bottom",
            fontsize=text_size, fontweight="bold")

# ---------- categorical plotting ----------
unique_vals = coverage['fclass'].unique()
colors = plt.cm.tab10.colors  # 10-color palette
color_map = {val: colors[i % len(colors)] for i, val in enumerate(unique_vals)}

fig, ax = plt.subplots(figsize=(12, 8))

for val, color in color_map.items():
    subset = coverage[coverage['fclass'] == val].to_crs(epsg=3857)
    subset.plot(ax=ax, color=color, linewidth=1, alpha=0.7, label=val)

# basemap
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, attribution_size=5, zorder=0)

# legend
handles = [mpatches.Patch(color=color, label=val) for val, color in color_map.items()]
ax.legend(handles=handles, title="Highway type", loc="lower right", bbox_to_anchor=(1, 0.1))

# scale + north arrow
add_scalebar(ax, loc="lower right")
add_north_arrow(ax, xy=(0.95, 0.85), size=0.08)

ax.set_axis_off()
plt.tight_layout()
plt.savefig('fig/TRAMS_coverage.png', dpi=300, bbox_inches='tight', pad_inches=0)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import contextily as ctx
import geopandas as gpd
import matplotlib.patches as mpatches
from pathlib import Path

# --- If you don't already have these helpers, uncomment this block ---
# from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
# from matplotlib.font_manager import FontProperties
# import math
# def _nice_length_m(data_width_m):
#     target = data_width_m / 5.0
#     exp = int(math.floor(math.log10(target))) if target > 0 else 0
#     base = target / (10 ** exp)
#     for b in (1, 2, 5, 10):
#         if base <= b:
#             return b * (10 ** exp)
#     return 10 * (10 ** exp)
# def add_scalebar(ax, loc="lower left", font_size=9, pad=0.2):
#     xmin, xmax = ax.get_xlim()
#     length_m = _nice_length_m(max(xmax - xmin, 1))
#     label = f"{int(length_m/1000)} km" if length_m >= 1000 else f"{int(length_m)} m"
#     sb = AnchoredSizeBar(ax.transData, length_m, label, loc,
#                          pad=pad, color='black', frameon=True,
#                          size_vertical=max((xmax - xmin), 1) * 0.003,
#                          fontproperties=FontProperties(size=font_size))
#     ax.add_artist(sb)
# def add_north_arrow(ax, xy=(0.08, 0.82), size=0.08, text="N", text_size=10):
#     ax.annotate("", xy=(xy[0], xy[1] + size), xytext=xy,
#                 xycoords="axes fraction", textcoords="axes fraction",
#                 arrowprops=dict(arrowstyle="-|>", linewidth=1.5, color="black"))
#     ax.text(xy[0], xy[1] + size + 0.015, text,
#             transform=ax.transAxes, ha="center", va="bottom",
#             fontsize=text_size, fontweight="bold")

# ---------- config ----------
pairs = [
    ("motorway",  "motorway_link"),
    ("trunk",     "trunk_link"),
    ("primary",   "primary_link"),
    ("secondary", "secondary_link"),
    ("tertiary",  "tertiary_link"),
]
main_color = "tab:blue"
link_color = "tab:orange"
outdir = Path("fig/")
outdir.mkdir(parents=True, exist_ok=True)

# ---------- prep ----------
rn_3857 = coverage.to_crs(3857)
all_bounds = rn_3857.total_bounds  # for consistent extent
xmin, ymin, xmax, ymax = all_bounds

for main, link in pairs:
    sub_main = rn_3857[rn_3857["fclass"] == main]
    sub_link = rn_3857[rn_3857["fclass"] == link]

    if sub_main.empty and sub_link.empty:
        # nothing to plot for this pair
        continue

    fig, ax = plt.subplots(figsize=(10, 7))

    # plot main first (thicker), then link (thinner)
    if not sub_main.empty:
        sub_main.plot(ax=ax, color=main_color, linewidth=1.4, alpha=0.9, zorder=3)
    if not sub_link.empty:
        sub_link.plot(ax=ax, color=link_color, linewidth=1.0, alpha=0.9, zorder=3)

    # basemap + consistent extent
    ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, attribution_size=5, zorder=0)
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_axis_off()

    # legend (inside map, lower-right; nudge up with bbox_to_anchor)
    handles = []
    if not sub_main.empty:
        handles.append(mpatches.Patch(color=main_color, label=main.title()))
    if not sub_link.empty:
        handles.append(mpatches.Patch(color=link_color, label=link.replace("_", " ").title()))
    ax.legend(handles=handles, title="Highway class",
              loc="lower right", bbox_to_anchor=(1.0, 0.20), frameon=True)

    # north arrow + scalebar
    add_north_arrow(ax, xy=(0.95, 0.85), size=0.08)
    add_scalebar(ax, loc="lower right")

    # title = f"{main.title()} & {link.replace('_', ' ').title()}"
    # ax.set_title(title, fontsize=13, pad=6)

    # save
    fname = f"coverage_pair_{main}.png"
    fig.savefig(outdir / fname, dpi=300, bbox_inches="tight", pad_inches=0)
    plt.show()
    plt.close(fig)


In [ ]:
missingness = coverage.isna().mean()
missingness.plot(kind='bar')
plt.title('Missingness of Road Network')
plt.xlabel('Attributes')
plt.ylabel('Missingness Proportion')
plt.show()

In [ ]:
# plot both accident and coverage map

# ---------- helpers ----------
def _nice_length_m(data_width_m):
    """Pick a nice scalebar length (1/2/5 x 10^n) ~ 1/5 of axis width."""
    target = data_width_m / 5.0
    import math
    exp = int(math.floor(math.log10(target))) if target > 0 else 0
    base = target / (10 ** exp)
    for b in (1, 2, 5, 10):
        if base <= b:
            return b * (10 ** exp)
    return 10 * (10 ** exp)

def add_scalebar(ax, loc="lower left", font_size=9, pad=0.2):
    """Add a scalebar (meters) to an EPSG:3857 axis."""
    xmin, xmax = ax.get_xlim()
    length_m = _nice_length_m(xmax - xmin)
    label = f"{int(length_m/1000)} km" if length_m >= 1000 else f"{int(length_m)} m"
    fp = FontProperties(size=font_size)
    sb = AnchoredSizeBar(ax.transData, length_m, label, loc,
                         pad=pad, color='black', frameon=True,
                         size_vertical=(xmax - xmin) * 0.003,
                         fontproperties=fp)
    ax.add_artist(sb)

def add_north_arrow(ax, xy=(0.8, 0.82), size=0.08, text="N", text_size=10):
    """Add a simple north arrow in axes fraction coords."""
    ax.annotate("", xy=(xy[0], xy[1] + size), xytext=xy,
                xycoords="axes fraction", textcoords="axes fraction",
                arrowprops=dict(arrowstyle="-|>", linewidth=1.5, color="black"))
    ax.text(xy[0], xy[1] + size + 0.015, text,
            transform=ax.transAxes, ha="center", va="bottom",
            fontsize=text_size, fontweight="bold")

# ---------- categorical plotting ----------
unique_vals = coverage['fclass'].unique()
colors = plt.cm.tab10.colors  # 10-color palette
color_map = {val: colors[i % len(colors)] for i, val in enumerate(unique_vals)}

fig, ax = plt.subplots(figsize=(12, 8))

for val, color in color_map.items():
    subset = coverage[coverage['fclass'] == val].to_crs(epsg=3857)
    subset.plot(ax=ax, color=color, linewidth=1, alpha=0.7, label=val)

# basemap
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, attribution_size=5, zorder=0)

# legend
handles = [mpatches.Patch(color=color, label=val) for val, color in color_map.items()]
ax.legend(handles=handles, title="Highway type", loc="lower right", bbox_to_anchor=(1, 0.1))

# scale + north arrow
add_scalebar(ax, loc="lower right")
add_north_arrow(ax, xy=(0.95, 0.85), size=0.08)


ax.set_axis_off()
plt.tight_layout()
plt.savefig('fig/TRAMS_coverage.png', dpi=300, bbox_inches='tight', pad_inches=0)
plt.show()


In [ ]:
import contextily as ctx
import matplotlib.pyplot as plt
import geopandas as gpd
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
from matplotlib.font_manager import FontProperties

# --- assume `coverage` is the HOTOSM roads GeoDataFrame (EPSG:4326)
# --- and `gdf_th` are the TRAMS crash points (EPSG:4326)

# Reproject once
coverage_3857 = coverage.to_crs(3857)
crashes_3857  = accidents.to_crs(3857)   # crash locations

# ---------- categorical plotting ----------
unique_vals = coverage_3857['fclass'].unique()
colors = plt.cm.tab10.colors  # 10-color palette
color_map = {val: colors[i % len(colors)] for i, val in enumerate(unique_vals)}

fig, ax = plt.subplots(figsize=(12, 8))

# plot roads by class
for val, color in color_map.items():
    subset = coverage_3857[coverage_3857['fclass'] == val]
    if not subset.empty:
        subset.plot(ax=ax, color=color, linewidth=1, alpha=0.7, zorder=2)

# basemap
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, attribution_size=5, zorder=0)

# --- overlay crash points ---
# small, slightly transparent red dots so they show on light basemap
crashes_3857.plot(ax=ax, markersize=.01, color="red", alpha=0.6, zorder=3)

# legend (roads + crash points)
road_handles = [mpatches.Patch(color=color, label=val) for val, color in color_map.items()]
crash_handle = Line2D([0], [0], marker='o', color='w', label='Crash location',
                      markerfacecolor='red', markersize=6)
handles = road_handles + [crash_handle]
ax.legend(handles=handles, title="Layers", loc="lower right", bbox_to_anchor=(1, 0.12), frameon=True)

# scale + north arrow
add_scalebar(ax, loc="lower right")
add_north_arrow(ax, xy=(0.95, 0.85), size=0.08)

ax.set_axis_off()
plt.tight_layout()
# plt.savefig('fig/TRAMS_coverage_with_crashes.png', dpi=300, bbox_inches='tight', pad_inches=0)
plt.show()


In [ ]:
provinces

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx
from matplotlib.patches import Patch

# --- inputs you already have ---
# provinces: GeoDataFrame of Thai provinces (EPSG:4326)
# coverage:  GeoDataFrame of OSM roads with column 'fclass' (EPSG:4326)
# gdf:       motorcycle crash points with geometry (EPSG:4326)

# Consistent CRS
provinces = provinces.to_crs(3857)
coverage  = coverage.to_crs(3857)
gdf       = accidents.to_crs(3857)

# Style for each road class
road_colors = {
    "trunk":          "#1f77b4",
    "trunk_link":     "#ff7f0e",
    "primary":        "#2ca02c",
    "primary_link":   "#d62728",
    "secondary":      "#9467bd",
    "secondary_link": "#8c564b",
    "tertiary":       "#e377c2",
    "tertiary_link":  "#7f7f7f",
}
road_widths = {
    "trunk": 1.6, "trunk_link": 1.2, "primary": 1.5, "primary_link": 1.1,
    "secondary": 1.2, "secondary_link": 1.0, "tertiary": 1.0, "tertiary_link": 0.9
}

def add_north(ax, xy=(0.06, 0.92), size=0.08):
    ax.annotate("", xy=(xy[0], xy[1]-size), xytext=xy, xycoords="axes fraction",
                arrowprops=dict(arrowstyle="-|>", color="black", lw=1.8))
    ax.text(xy[0]+0.005, xy[1]+0.01, "N", transform=ax.transAxes,
            ha="left", va="bottom", fontsize=10, fontweight="bold")

def add_scalebar_mercator(ax, loc="lower left"):
    from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
    from matplotlib.font_manager import FontProperties
    xmin, xmax = ax.get_xlim()
    length = (xmax - xmin) / 5.0  # ~1/5 width
    label = f"{int(length/1000)} km"
    fp = FontProperties(size=9)
    sb = AnchoredSizeBar(ax.transData, length, label, loc,
                         pad=0.2, color='black', frameon=True,
                         size_vertical=(xmax-xmin)*0.003, fontproperties=fp)
    ax.add_artist(sb)

def plot_city(ax, province_name, point_size=5):
    # province bbox (no clipping)
    prov = provinces[provinces["PROV_NAME"].str.contains(province_name, case=False, na=False)]
    if prov.empty:
        prov = provinces[provinces["PROV_NAME"].str.contains(province_name, case=False, na=False)]
    if prov.empty:
        raise ValueError(f"Province '{province_name}' not found in attributes.")

    minx, miny, maxx, maxy = prov.total_bounds
    pad_x = (maxx - minx) * 0.03
    pad_y = (maxy - miny) * 0.03
    ax.set_xlim(minx - pad_x, maxx + pad_x)
    ax.set_ylim(miny - pad_y, maxy + pad_y)

    # basemap at bottom
    ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, attribution_size=5, zorder=0)

    # draw roads (only those intersecting bbox, but not clipping)
    bbox = gpd.GeoSeries([prov.unary_union.envelope], crs=provinces.crs)
    roads_view = coverage[coverage.intersects(bbox.geometry.iloc[0].buffer(0))]

    for cls, color in road_colors.items():
        seg = roads_view[roads_view["fclass"] == cls]
        if not seg.empty:
            seg.plot(ax=ax, color=color, linewidth=road_widths[cls], alpha=0.9, zorder=2)

    # draw province outline with no fill (this avoids grey blocks)
    prov.boundary.plot(ax=ax, color="grey", linewidth=1.0, zorder=3)

    # crash points within bbox
    pts_view = gdf[gdf.intersects(bbox.geometry.iloc[0])]
    if not pts_view.empty:
        pts_view.plot(
            ax=ax,
            color="black",
            markersize=1,   # adjust size for clarity
            alpha=0.4,
            marker="x",     # change to "*" if you prefer stars
            zorder=4
        )
    # pts_view = gdf[gdf.intersects(bbox.geometry.iloc[0])]
    # if not pts_view.empty:
    #     pts_view.plot(ax=ax, color="grey", markersize=.5, alpha=0.2, zorder=4)

    # legend
    handles = [Patch(facecolor=road_colors[k], edgecolor='none', label=k.replace("_", " "))
               for k in road_colors]
    handles.append(Patch(facecolor="crimson", edgecolor="none", label="Crash location"))
    leg = ax.legend(handles=handles, title=province_name, loc="lower right",
                    frameon=True, fontsize=8)
    leg.get_frame().set_alpha(0.9)

    add_north(ax, xy=(0.06, 0.96), size=0.09)
    add_scalebar_mercator(ax, loc="lower left")
    ax.set_axis_off()

# ---- Make the two maps ----
fig, axes = plt.subplots(1, 2, figsize=(14, 8), constrained_layout=True)

plot_city(axes[0], "Bangkok")      # or "Bangkok Metropolis" depending on the shapefile field
axes[0].set_title("Bangkok", fontsize=12)

plot_city(axes[1], "Chiang Mai")
axes[1].set_title("Chiang Mai", fontsize=12)

plt.show()


In [ ]:
coverage

# Lixelise

In [ ]:
import geopandas as gpd
import numpy as np
from shapely.ops import substring

In [ ]:
gdf = coverage.to_crs("EPSG:32647")  # UTM 47N (meters), if not already
gdf["len_m"] = gdf.geometry.length

print("count:", len(gdf))
print("min/median/mean/max:", gdf["len_m"].min(), gdf["len_m"].median(), gdf["len_m"].mean(), gdf["len_m"].max())
print("quantiles (m):", gdf["len_m"].quantile([0.1,0.25,0.5,0.75,0.9,0.95,0.99]).to_dict())


In [ ]:
gdf = coverage.to_crs(32647)               # UTM 47N (meters)
gdf = gdf.explode(index_parts=False)      # break MultiLineStrings
gdf = gdf[gdf.geom_type == "LineString"]  # keep only LineStrings
gdf = gdf[gdf.length > 0]                 # drop zero-length

In [ ]:
def split_line_equal(line, target=100.0, min_len=50.0):
    L = float(line.length)
    # If the line is short, keep it as-is
    if L <= target + min_len:
        return [line]

    # initial number of ~target segments
    k = max(2, int(np.floor(L / target)))  # at least 2
    seg_len = L / k
    # if last piece would be too small, reduce k until remainder is acceptable
    # (this makes each segment a bit longer than target, but avoids tiny tails)
    while k > 1 and (L - (k-1)*seg_len) < min_len:
        k -= 1
        seg_len = L / k

    # cut at equal spacing: seg_len, 2*seg_len, ..., (k-1)*seg_len
    cuts = [i * seg_len for i in range(1, k)]
    parts = []
    prev = 0.0
    for c in cuts:
        parts.append(substring(line, prev, c, normalized=False))
        prev = c
    parts.append(substring(line, prev, L, normalized=False))
    return parts

def lixelize_equal(gdf, target=100.0, min_len=50.0, id_col=None):
    # 0) ensure projected CRS in metres
    if gdf.crs is None or not gdf.crs.is_projected:
        raise ValueError("Project the data to a metric CRS (e.g., UTM) before lixelising.")
    # 1) explode MULTILINESTRING to LineString
    gdf_ls = gdf.explode(index_parts=False)
    gdf_ls = gdf_ls[gdf_ls.geometry.geom_type == "LineString"].copy()

    cols = [c for c in gdf_ls.columns if c != "geometry"]
    out = []
    for idx, row in gdf_ls.iterrows():
        line = row.geometry
        parts = split_line_equal(line, target=target, min_len=min_len)
        for order, p in enumerate(parts):
            rec = {c: row[c] for c in cols}
            if id_col and id_col in row:
                rec["parent_id"] = row[id_col]
            rec["order"] = order
            rec["geometry"] = p
            out.append(rec)

    return gpd.GeoDataFrame(out, crs=gdf.crs)

# ---- usage ----
# roads = roads.to_crs(32647)  # make sure metres; the UTM 47N is fine
# lixels_100 = lixelize_equal(gdf, target=100.0, min_len=50.0, id_col="osm_id")
# lixels_100["len_lixel_m"] = lixels_100.geometry.length
# print(lixels_100["len_lixel_m"].describe())


In [ ]:
lixels_50 = lixelize_equal(gdf, target=50.0, min_len=25.0, id_col="osm_id")
lixels_100 = lixelize_equal(gdf, target=100.0, min_len=50.0, id_col="osm_id")
lixels_200 = lixelize_equal(gdf, target=200.0, min_len=50.0, id_col="osm_id")
lixels_300 = lixelize_equal(gdf, target=300.0, min_len=50.0, id_col="osm_id")
lixels_500 = lixelize_equal(gdf, target=500.0, min_len=50.0, id_col="osm_id")

In [ ]:
lixels_50['lixel_len_m'] = lixels_50.geometry.length
lixels_100['lixel_len_m'] = lixels_100.geometry.length
lixels_200['lixel_len_m'] = lixels_200.geometry.length
lixels_300['lixel_len_m'] = lixels_300.geometry.length
lixels_500['lixel_len_m'] = lixels_500.geometry.length

In [ ]:
# histogram of lixel lengths
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 6))
plt.hist(lixels_50['lixel_len_m'], bins=30, alpha=0.5, label='50m target')
plt.hist(lixels_100['lixel_len_m'], bins=30, alpha=0.5, label='100m target')
plt.hist(lixels_200['lixel_len_m'], bins=30, alpha=0.5, label='200m target')
plt.hist(lixels_300['lixel_len_m'], bins=30, alpha=0.5, label='300m target')
plt.hist(lixels_500['lixel_len_m'], bins=30, alpha=0.5, label='500m target')
plt.xlabel('Lixel Length (m)')
plt.ylabel('Frequency')
plt.title('Histogram of Lixel Lengths for Different Target Lengths')
plt.legend()
plt.show()

In [ ]:
# save all lixels to file
lixels_50.to_file("roadnetwork/tha_highway_lixels_50m.gpkg", layer="lixels_50m", driver="GPKG")
lixels_100.to_file("roadnetwork/tha_highway_lixels_100m.gpkg", layer="lixels_100m", driver="GPKG")
lixels_200.to_file("roadnetwork/tha_highway_lixels_200m.gpkg", layer="lixels_200m", driver="GPKG")
lixels_300.to_file("roadnetwork/tha_highway_lixels_300m.gpkg", layer="lixels_300m", driver="GPKG")
lixels_500.to_file("roadnetwork/tha_highway_lixels_500m.gpkg", layer="lixels_500m", driver="GPKG")

In [ ]:
lixels_50 = gpd.read_file('../data/roadnetwork/tha_highway_lixels_50m.gpkg', driver="GPKG")
lixels_100 = gpd.read_file('../data/roadnetwork/tha_highway_lixels_100m.gpkg', driver="GPKG")
lixels_200 = gpd.read_file('../data/roadnetwork/tha_highway_lixels_200m.gpkg', driver="GPKG")
lixels_300 = gpd.read_file('../data/roadnetwork/tha_highway_lixels_300m.gpkg', driver="GPKG")
lixels_500 = gpd.read_file('../data/roadnetwork/tha_highway_lixels_500m.gpkg', driver="GPKG")

In [ ]:
lixels_50 = gpd.read_file('../data/roadnetwork/tha_highway_lixels_50m.gpkg', driver="GPKG")


In [ ]:
lixels_50['nkde_sample_id'] = lixels_50.index.astype(int) +1
lixels_100['nkde_sample_id'] = lixels_100.index.astype(int) +1
lixels_200['nkde_sample_id'] = lixels_200.index.astype(int) +1
lixels_300['nkde_sample_id'] = lixels_300.index.astype(int) +1
lixels_500['nkde_sample_id'] = lixels_500.index.astype(int) +1

In [ ]:
len(lixels_50), len(lixels_100), len(lixels_200), len(lixels_300), len(lixels_500)


In [ ]:
max(lixels_50['nkde_sample_id']), max(lixels_100['nkde_sample_id']), max(lixels_200['nkde_sample_id']), max(lixels_300['nkde_sample_id']), max(lixels_500['nkde_sample_id'])

## Samples

In [ ]:
def lixel_midpoints(lixels_gdf, id_col=None):
    """
    Create one point per lixel at the midpoint along the line (on-geometry).

    Parameters
    ----------
    lixels_gdf : GeoDataFrame
        LineString lixels in a metric CRS (e.g., UTM).
    id_col : str, optional
        Column name for the parent/road id to carry through.

    Returns
    -------
    GeoDataFrame
        Points at mid-arc-length for each lixel with length attributes.
    """
    if lixels_gdf.crs is None or not lixels_gdf.crs.is_projected:
        raise ValueError("Project to a metric CRS (metres) before creating midpoints.")

    out = lixels_gdf.copy()
    out["lixel_len_m"] = out.geometry.length

    # Midpoint along arc length (guaranteed on the line)
    mid_geom = out.geometry.interpolate(0.5, normalized=True)

    # Build points GDF (carry all attrs, replace geometry)
    pts = gpd.GeoDataFrame(out.drop(columns="geometry"), geometry=mid_geom, crs=out.crs)

    # Optional convenience fields
    pts["s_mid_m"] = pts["lixel_len_m"] * 0.5
    if id_col and id_col in lixels_gdf.columns:
        pts.rename(columns={id_col: "parent_id"}, inplace=True)

    # handy unique id for samples
    pts["nkde_sample_id"] = range(1, len(pts) + 1)
    return pts

# ---- usage ----
# lixels_100 is the lixel GeoDataFrame (LineString, metric CRS)
# midpoints = lixel_midpoints(lixels_100, id_col="parent_id")
# print(midpoints.head())

# save if needed
# midpoints.to_file("nkde_samples_midpoints.gpkg", layer="nkde_samples", driver="GPKG")


In [ ]:
sample_100 = lixel_midpoints(lixels_100, id_col="parent_id")
sample_50 = lixel_midpoints(lixels_50, id_col="parent_id")
sample_200 = lixel_midpoints(lixels_200, id_col="parent_id")
sample_300 = lixel_midpoints(lixels_300, id_col="parent_id")
sample_500 = lixel_midpoints(lixels_500, id_col="parent_id")

In [ ]:
# save samples to file
sample_50.to_file("roadnetwork/tha_highway_nkde_samples_50m.gpkg", layer="nkde_samples_50m", driver="GPKG")
sample_100.to_file("roadnetwork/tha_highway_nkde_samples_100m.gpkg", layer="nkde_samples_100m", driver="GPKG")
sample_200.to_file("roadnetwork/tha_highway_nkde_samples_200m.gpkg", layer="nkde_samples_200m", driver="GPKG")
sample_300.to_file("roadnetwork/tha_highway_nkde_samples_300m.gpkg", layer="nkde_samples_300m", driver="GPKG")
sample_500.to_file("roadnetwork/tha_highway_nkde_samples_500m.gpkg", layer="nkde_samples_500m", driver="GPKG")


In [ ]:
sample_50 = gpd.read_file('../data/roadnetwork/tha_highway_nkde_samples_50m.gpkg', driver="GPKG")
sample_100 = gpd.read_file('../data/roadnetwork/tha_highway_nkde_samples_100m.gpkg', driver="GPKG")
sample_200 = gpd.read_file('../data/roadnetwork/tha_highway_nkde_samples_200m.gpkg', driver="GPKG")
sample_300 = gpd.read_file('../data/roadnetwork/tha_highway_nkde_samples_300m.gpkg', driver="GPKG")
sample_500 = gpd.read_file('../data/roadnetwork/tha_highway_nkde_samples_500m.gpkg', driver="GPKG")

In [ ]:
len(sample_50), len(sample_100), len(sample_200), len(sample_300), len(sample_500)


In [ ]:
max(sample_50['nkde_sample_id']), max(sample_100['nkde_sample_id']), max(sample_200['nkde_sample_id']), max(sample_300['nkde_sample_id']), max(sample_500['nkde_sample_id'])